In [ ]:
# ==== CELL 1: Install dependencies ====
# sentence-transformers pulls in torch/transformers as needed. Colab usually
# already has torch installed with the right GPU build, so we don't pin it
# here to avoid fighting Colab's preinstalled version.
!pip install -q sentence-transformers

In [ ]:
# ==== CELL 2: Imports ====
import json
import random
from itertools import combinations
from pathlib import Path

from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses,
    evaluation,
)
from torch.utils.data import DataLoader

random.seed(42)  # reproducible pair sampling / train-eval split

/tmp/ipykernel_579/3870825593.py:7: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import (
/tmp/ipykernel_579/3870825593.py:7: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers import (


In [ ]:
# ==== CELL 3: Upload training_data_synthetic.json ====
# Run this cell, then use the file picker to upload the JSON file you
# generated locally with scripts/generate_synthetic_training_data.py.
from google.colab import files

uploaded = files.upload()
DATA_PATH = next(iter(uploaded.keys()))  # grabs whatever filename was uploaded


Saving training_data_synthetic.json to training_data_synthetic.json


In [ ]:
# ==== CELL 4: Load and inspect the data ====
with open(DATA_PATH, "r", encoding="utf-8") as f:
    records = json.load(f)

print(f"Loaded {len(records)} records")

# Group by category up front — this grouping is what drives both the
# training pair construction (Cell 5) and the eval triplet construction
# (Cell 7) below.
by_category: dict[str, list[dict]] = {}
for r in records:
    by_category.setdefault(r["category"], []).append(r)

for category, items in by_category.items():
    print(f"  {category}: {len(items)} records")

Loaded 360 records
  Web Development: 60 records
  Data Science: 60 records
  Design: 60 records
  Cloud & DevOps: 60 records
  Marketing: 60 records
  Backend & Systems: 60 records


In [ ]:
# ==== CELL 5: Train/eval split (per category, so both splits cover every category) ====
# Held-out records are used ONLY for building the eval triplets in Cell 7,
# never for training pairs — otherwise the eval score wouldn't tell us
# anything about generalization.
EVAL_FRACTION = 0.15

train_by_category: dict[str, list[dict]] = {}
eval_by_category: dict[str, list[dict]] = {}

for category, items in by_category.items():
    shuffled = items[:]
    random.shuffle(shuffled)
    split_idx = max(1, int(len(shuffled) * (1 - EVAL_FRACTION)))
    train_by_category[category] = shuffled[:split_idx]
    eval_by_category[category] = shuffled[split_idx:]

print("Train/eval split sizes per category:")
for category in by_category:
    print(f"  {category}: train={len(train_by_category[category])}, eval={len(eval_by_category[category])}")

Train/eval split sizes per category:
  Web Development: train=51, eval=9
  Data Science: train=51, eval=9
  Design: train=51, eval=9
  Cloud & DevOps: train=51, eval=9
  Marketing: train=51, eval=9
  Backend & Systems: train=51, eval=9


In [ ]:
# ==== CELL 6: Build training pairs ====
# Two kinds of positive pairs, both fed to MultipleNegativesRankingLoss:
#
#   (a) Self pairs: (title, description) of the SAME course. Strongest
#       possible positive signal — teaches the model that a course's title
#       and its own description should embed close together.
#
#   (b) Category pairs: (course_A_text, course_B_text) for two DIFFERENT
#       courses in the SAME category. Teaches category-level clustering,
#       which is what "similar courses" recommendations actually rely on.
#
# We deliberately don't need explicit negative pairs — MultipleNegativesRankingLoss
# treats every other example in the same training batch as an implicit
# negative, which is exactly why batches should mix categories (the default
# DataLoader shuffle handles this).

def course_text(record: dict) -> str:
    """Combines title + description into one embedding input string."""
    return f"{record['title']}. {record['description']}"


train_examples: list[InputExample] = []

for category, items in train_by_category.items():
    # (a) self pairs
    for r in items:
        train_examples.append(InputExample(texts=[r["title"], r["description"]]))

    # (b) category pairs — cap how many cross-pairs we draw per category so
    # a category with many records doesn't drown out smaller ones. Sampling
    # a fixed number of random combinations rather than all C(n,2) pairs
    # keeps this linear instead of quadratic as your dataset grows.
    all_pairs = list(combinations(items, 2))
    random.shuffle(all_pairs)
    MAX_CATEGORY_PAIRS = 200
    for a, b in all_pairs[:MAX_CATEGORY_PAIRS]:
        train_examples.append(InputExample(texts=[course_text(a), course_text(b)]))

print(f"Built {len(train_examples)} training pairs")

Built 1506 training pairs


In [ ]:
# ==== CELL 7: Build evaluation triplets ====
# TripletEvaluator scores the model on (anchor, positive, negative) triplets:
# positive = another course in the SAME category as the anchor (held-out
# split), negative = a course from a DIFFERENT category. The reported metric
# is the fraction of triplets where the model correctly embeds the anchor
# closer to the positive than the negative — a direct proxy for "will
# same-category courses actually rank as more similar than different-category
# ones" once this ships to Phase 3.

categories = list(eval_by_category.keys())
anchors, positives, negatives = [], [], []

for category in categories:
    eval_items = eval_by_category[category]
    if len(eval_items) < 2:
        continue  # need at least 2 to form an anchor/positive pair

    other_categories = [c for c in categories if c != category]

    for i in range(len(eval_items) - 1):
        anchor = eval_items[i]
        positive = eval_items[i + 1]
        negative_category = random.choice(other_categories)
        negative = random.choice(eval_by_category[negative_category])

        anchors.append(course_text(anchor))
        positives.append(course_text(positive))
        negatives.append(course_text(negative))

print(f"Built {len(anchors)} evaluation triplets")

triplet_evaluator = evaluation.TripletEvaluator(
    anchors=anchors,
    positives=positives,
    negatives=negatives,
    name="course-category-triplets",
)

Built 48 evaluation triplets


In [ ]:
# ==== CELL 8: Load base model ====
BASE_MODEL_NAME = "all-MiniLM-L6-v2"  # 384-dim output, matches our pgvector column plan for Phase 3
model = SentenceTransformer(BASE_MODEL_NAME)


# ==== CELL 9: Baseline eval (before fine-tuning) ====
# Worth checking this score before training — it's the number fine-tuning
# needs to beat. If it's already very high, the synthetic data may be too
# easy/separable to teach the model much.
baseline_score = triplet_evaluator(model)
print(f"Baseline (pre-fine-tune) triplet accuracy: {baseline_score}")


# ==== CELL 10: Fine-tune ====
BATCH_SIZE = 32
EPOCHS = 2
WARMUP_STEPS = int(len(train_examples) / BATCH_SIZE * EPOCHS * 0.1)  # ~10% warmup, standard default

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss = losses.MultipleNegativesRankingLoss(model)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=triplet_evaluator,
    evaluation_steps=50,
    epochs=EPOCHS,
    warmup_steps=WARMUP_STEPS,
    output_path="fine_tuned_course_embedder",  # best checkpoint (by eval score) saved here automatically
    show_progress_bar=True,
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Baseline (pre-fine-tune) triplet accuracy: {'course-category-triplets_cosine_accuracy': 0.7291666865348816}


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Course-category-triplets Cosine Accuracy
48,No log,No log,0.895833
50,No log,No log,0.895833
96,No log,No log,1.000000
100,No log,No log,1.000000
144,No log,No log,1.000000
150,No log,No log,1.000000
192,No log,No log,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# ==== CELL 11: Confirm final eval score ====
final_score = triplet_evaluator(model)
print(f"Baseline triplet accuracy: {baseline_score}")
print(f"Fine-tuned triplet accuracy: {final_score}")
# If final_score isn't meaningfully above baseline, the likely fixes are:
# more synthetic records per category (RECORDS_PER_CATEGORY in the generator
# script), more epochs, or richer subtopic/template variety.

Baseline triplet accuracy: {'course-category-triplets_cosine_accuracy': 0.7291666865348816}
Fine-tuned triplet accuracy: {'course-category-triplets_cosine_accuracy': 1.0}


In [ ]:
# ==== CELL 12: Zip and download the fine-tuned model ====
import shutil

shutil.make_archive("fine_tuned_course_embedder", "zip", "fine_tuned_course_embedder")
files.download("fine_tuned_course_embedder.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>